In [0]:
%sql

-- First data ingestion process, full load from bronze, filter products and deduplicate


-- CREATE OR REPLACE TABLE retail_lakehouse_dr.silver.products
-- USING DELTA
-- AS
-- SELECT
--     TRY_CAST(get_json_object(payload_json, '$.id') AS INT) AS product_id,
--     TRIM(get_json_object(payload_json, '$.title')) AS product_name,
--     LOWER(TRIM(get_json_object(payload_json, '$.category'))) AS category,
--     NULLIF(TRIM(get_json_object(payload_json, '$.brand')), '') AS brand,
--     TRY_CAST(get_json_object(payload_json, '$.price') AS DECIMAL(12, 2)) AS price,
--     TRY_CAST(get_json_object(payload_json, '$.discountPercentage') AS DECIMAL(5, 2)) AS discount_percentage,
--     TRY_CAST(get_json_object(payload_json, '$.rating') AS DECIMAL(4, 2)) AS rating,
--     TRY_CAST(get_json_object(payload_json, '$.stock') AS INT) AS stock,
--     get_json_object(payload_json, '$.sku') AS sku,
--     get_json_object(payload_json, '$.availabilityStatus') AS availability_status,
--     TRY_CAST(get_json_object(payload_json, '$.minimumOrderQuantity') AS INT) AS minimum_order_quantity,

--     source,
--     record_id AS source_record_id,
--     run_id,
--     TO_TIMESTAMP(extracted_at) AS extracted_at

-- FROM retail_lakehouse_dr.bronze.api_snapshots
-- WHERE source IN ('products', 'products_simulated');

In [0]:
%sql

-- Insert data into silver products, incremental load from bronze, filter products and deduplicate

MERGE INTO retail_lakehouse_dr.silver.products AS target
USING (
    SELECT
        TRY_CAST(get_json_object(payload_json, '$.id') AS INT) AS product_id,
        TRIM(get_json_object(payload_json, '$.title')) AS product_name,
        LOWER(TRIM(get_json_object(payload_json, '$.category'))) AS category,
        NULLIF(TRIM(get_json_object(payload_json, '$.brand')), '') AS brand,
        TRY_CAST(get_json_object(payload_json, '$.price') AS DECIMAL(12, 2)) AS price,
        TRY_CAST(get_json_object(payload_json, '$.discountPercentage') AS DECIMAL(5, 2)) AS discount_percentage,
        TRY_CAST(get_json_object(payload_json, '$.rating') AS DECIMAL(4, 2)) AS rating,
        TRY_CAST(get_json_object(payload_json, '$.stock') AS INT) AS stock,
        get_json_object(payload_json, '$.sku') AS sku,
        get_json_object(payload_json, '$.availabilityStatus') AS availability_status,
        TRY_CAST(get_json_object(payload_json, '$.minimumOrderQuantity') AS INT) AS minimum_order_quantity,
        source,
        record_id AS source_record_id,
        run_id,
        TO_TIMESTAMP(extracted_at) AS extracted_at
    FROM retail_lakehouse_dr.bronze.api_snapshots
    WHERE source IN ('products', 'products_simulated')
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY source, record_id, run_id
        ORDER BY TO_TIMESTAMP(extracted_at) DESC
    ) = 1
) AS incoming
ON  target.source = incoming.source
AND target.source_record_id = incoming.source_record_id
AND target.run_id = incoming.run_id

WHEN MATCHED AND NOT (
       target.product_id              <=> incoming.product_id
   AND target.product_name            <=> incoming.product_name
   AND target.category                <=> incoming.category
   AND target.brand                   <=> incoming.brand
   AND target.price                   <=> incoming.price
   AND target.discount_percentage     <=> incoming.discount_percentage
   AND target.rating                  <=> incoming.rating
   AND target.stock                   <=> incoming.stock
   AND target.sku                     <=> incoming.sku
   AND target.availability_status     <=> incoming.availability_status
   AND target.minimum_order_quantity  <=> incoming.minimum_order_quantity
   AND target.extracted_at            <=> incoming.extracted_at
)
THEN UPDATE SET *

WHEN NOT MATCHED THEN INSERT *;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
0,0,0,0


In [0]:
%sql

-- First data ingestion process, full load from bronze, filter carts and deduplicate


-- CREATE OR REPLACE TABLE retail_lakehouse_dr.silver.carts
-- USING DELTA
-- AS
-- SELECT
--     TRY_CAST(get_json_object(payload_json, '$.id') AS INT) AS cart_id,
--     TRY_CAST(get_json_object(payload_json, '$.userId') AS INT) AS customer_id,

--     TRY_CAST(get_json_object(payload_json, '$.total') AS DECIMAL(14, 2)) AS total,
--     TRY_CAST(get_json_object(payload_json, '$.discountedTotal') AS DECIMAL(14, 2)) AS discounted_total,
--     TRY_CAST(get_json_object(payload_json, '$.totalProducts') AS INT) AS total_products,
--     TRY_CAST(get_json_object(payload_json, '$.totalQuantity') AS INT) AS total_quantity,

--     get_json_object(payload_json, '$.products') AS products_json,

--     source,
--     record_id AS source_record_id,
--     run_id,
--     TO_TIMESTAMP(extracted_at) AS extracted_at

-- FROM retail_lakehouse_dr.bronze.api_snapshots
-- WHERE source IN ('carts', 'carts_simulated');


-----------------------------------------------------
-- ALTER TABLE retail_lakehouse_dr.silver.carts
-- ADD COLUMNS (cart_key STRING);

In [0]:
%sql

-- Insert data into silver carts, incremental load from bronze, filter carts and deduplicate

MERGE INTO retail_lakehouse_dr.silver.carts AS target
USING (
    SELECT
        TRY_CAST(get_json_object(payload_json, '$.id') AS INT) AS cart_id,
        TRY_CAST(get_json_object(payload_json, '$.userId') AS INT) AS customer_id,
        TRY_CAST(get_json_object(payload_json, '$.total') AS DECIMAL(14, 2)) AS total,
        TRY_CAST(get_json_object(payload_json, '$.discountedTotal') AS DECIMAL(14, 2)) AS discounted_total,
        TRY_CAST(get_json_object(payload_json, '$.totalProducts') AS INT) AS total_products,
        TRY_CAST(get_json_object(payload_json, '$.totalQuantity') AS INT) AS total_quantity,
        get_json_object(payload_json, '$.products') AS products_json,

        source,
        record_id AS source_record_id,
        run_id,
        TO_TIMESTAMP(extracted_at) AS extracted_at,

        CASE
            WHEN source = 'carts_simulated' THEN record_id
            ELSE COALESCE(
                CAST(TRY_CAST(get_json_object(payload_json, '$.id') AS INT) AS STRING),
                record_id
            )
        END AS cart_key

    FROM retail_lakehouse_dr.bronze.api_snapshots
    WHERE source IN ('carts', 'carts_simulated')

    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY source, record_id, run_id
        ORDER BY TO_TIMESTAMP(extracted_at) DESC
    ) = 1
) AS incoming

ON  target.source = incoming.source
AND target.source_record_id = incoming.source_record_id
AND target.run_id = incoming.run_id

WHEN MATCHED AND NOT (
       target.cart_id             <=> incoming.cart_id
   AND target.customer_id         <=> incoming.customer_id
   AND target.total               <=> incoming.total
   AND target.discounted_total    <=> incoming.discounted_total
   AND target.total_products      <=> incoming.total_products
   AND target.total_quantity      <=> incoming.total_quantity
   AND target.products_json       <=> incoming.products_json
   AND target.extracted_at        <=> incoming.extracted_at
   AND target.cart_key            <=> incoming.cart_key
)
THEN UPDATE SET
    target.cart_id          = incoming.cart_id,
    target.customer_id      = incoming.customer_id,
    target.total            = incoming.total,
    target.discounted_total = incoming.discounted_total,
    target.total_products   = incoming.total_products,
    target.total_quantity   = incoming.total_quantity,
    target.products_json    = incoming.products_json,
    target.extracted_at     = incoming.extracted_at,
    target.cart_key         = incoming.cart_key

WHEN NOT MATCHED THEN INSERT (
    cart_id,
    customer_id,
    total,
    discounted_total,
    total_products,
    total_quantity,
    products_json,
    source,
    source_record_id,
    run_id,
    extracted_at,
    cart_key
)
VALUES (
    incoming.cart_id,
    incoming.customer_id,
    incoming.total,
    incoming.discounted_total,
    incoming.total_products,
    incoming.total_quantity,
    incoming.products_json,
    incoming.source,
    incoming.source_record_id,
    incoming.run_id,
    incoming.extracted_at,
    incoming.cart_key
);

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
5,0,0,5


In [0]:
%sql

-- Cart items (dimension) table creation from carts
-- It preserves the granularity of each cart per execution (run_id)

-- CREATE TABLE IF NOT EXISTS retail_lakehouse_dr.silver.cart_items (
--     cart_id INT,
--     customer_id INT,
--     product_id INT,
--     product_name STRING,
--     price DECIMAL(12, 2),
--     quantity INT,
--     total DECIMAL(14, 2),
--     discount_percentage DECIMAL(5, 2),
--     discounted_total DECIMAL(14, 2),
--     thumbnail STRING,
--     source STRING,
--     cart_source_record_id STRING,
--     run_id STRING,
--     extracted_at TIMESTAMP
-- )
-- USING DELTA;


-----------------------------------------------------
-- ALTER TABLE retail_lakehouse_dr.silver.cart_items
-- ADD COLUMNS (cart_key STRING);


In [0]:

%sql

-- Insert cart items while preserving each item's position in products_json.
-- item_index identifies the line within each cart snapshot.

MERGE INTO retail_lakehouse_dr.silver.cart_items AS target

USING (
    SELECT
        c.cart_id,
        c.cart_key,
        c.customer_id,

        COALESCE(item.id, item.productId) AS product_id,

        item.title AS product_name,
        item.price,
        item.quantity,
        item.total,
        item.discountPercentage AS discount_percentage,
        item.discountedTotal AS discounted_total,
        item.thumbnail,

        pos AS item_index,

        c.source,
        c.source_record_id AS cart_source_record_id,
        c.run_id,
        TO_TIMESTAMP(c.extracted_at) AS extracted_at

    FROM retail_lakehouse_dr.silver.carts AS c

    LATERAL VIEW POSEXPLODE(
        FROM_JSON(
            c.products_json,
            'ARRAY<STRUCT<
                id: INT,
                productId: INT,
                title: STRING,
                price: DECIMAL(12,2),
                quantity: INT,
                total: DECIMAL(14,2),
                discountPercentage: DECIMAL(5,2),
                discountedTotal: DECIMAL(14,2),
                thumbnail: STRING
            >>'
        )
    ) exploded AS pos, item

    WHERE COALESCE(item.id, item.productId) IS NOT NULL

    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY
            c.source,
            c.source_record_id,
            c.run_id,
            pos
        ORDER BY TO_TIMESTAMP(c.extracted_at) DESC
    ) = 1

) AS incoming

ON  target.source = incoming.source
AND target.cart_source_record_id = incoming.cart_source_record_id
AND target.run_id = incoming.run_id
AND target.item_index = incoming.item_index

WHEN MATCHED AND NOT (
       target.cart_id <=> incoming.cart_id
   AND target.cart_key <=> incoming.cart_key
   AND target.customer_id <=> incoming.customer_id
   AND target.product_id <=> incoming.product_id
   AND target.product_name <=> incoming.product_name
   AND target.price <=> incoming.price
   AND target.quantity <=> incoming.quantity
   AND target.total <=> incoming.total
   AND target.discount_percentage <=> incoming.discount_percentage
   AND target.discounted_total <=> incoming.discounted_total
   AND target.thumbnail <=> incoming.thumbnail
   AND target.extracted_at <=> incoming.extracted_at
)
THEN UPDATE SET *

WHEN NOT MATCHED THEN INSERT *;


num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
14,0,0,14
